# Airline Passenger Satisfaction Analysis & Prediction
## End-to-End Exploratory Data Analysis and Logistic Regression Modeling Pipeline

### Project Overview:
This notebook documents the end-to-end machine learning lifecycle for the **Airline Passenger Satisfaction** dataset:
1. **Library Imports & Setup**
2. **Data Ingestion & Integrity Checks**
3. **Data Cleaning & Missing Value Assessment**
4. **Comprehensive Exploratory Data Analysis (EDA)**
5. **Preprocessing Pipeline (Imputation, Scaling, One-Hot Encoding)**
6. **Logistic Regression Classifier Training**
7. **Model Evaluation & Metric Interpretation (Accuracy, F1, ROC-AUC, Confusion Matrix)**
8. **Feature Importance & Coefficient Analysis**
9. **Exporting Trained Model Pipeline with Joblib**


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, roc_curve
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
print('Libraries imported successfully.')


### 2. Loading Existing Datasets
We load the pre-existing training (`train.csv`) and testing (`test.csv`) sets without data duplication or modification.


In [ ]:
train_path = 'train.csv' if os.path.exists('train.csv') else os.path.join('..', 'train.csv')
test_path = 'test.csv' if os.path.exists('test.csv') else os.path.join('..', 'test.csv')

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f'Train shape: {train_df.shape}')
print(f'Test shape:  {test_df.shape}')
train_df.head()


### 3. Understanding the Dataset & Handling Missing Values
Inspecting data types, non-predictive identifiers, and null values.


In [ ]:
print('Train missing values:')
print(train_df.isnull().sum()[train_df.isnull().sum() > 0])

print('\nTest missing values:')
print(test_df.isnull().sum()[test_df.isnull().sum() > 0])

print('\nTarget distribution (Train):')
print(train_df['satisfaction'].value_counts(normalize=True))


### 4. Exploratory Data Analysis (EDA)
Visualizing key drivers: class, passenger type, and service satisfaction ratings.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall target distribution
sns.countplot(data=train_df, x='satisfaction', ax=axes[0], palette=['#f43f5e', '#10b981'])
axes[0].set_title('Passenger Satisfaction Distribution')
axes[0].set_ylabel('Passenger Count')

# Satisfaction by Travel Class
sns.countplot(data=train_df, x='Class', hue='satisfaction', ax=axes[1], palette=['#f43f5e', '#10b981'])
axes[1].set_title('Satisfaction by Travel Class')
axes[1].set_ylabel('Passenger Count')

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Wi-Fi service vs satisfaction
sns.countplot(data=train_df, x='Inflight wifi service', hue='satisfaction', ax=axes[0], palette=['#f43f5e', '#10b981'])
axes[0].set_title('Satisfaction by Inflight Wi-Fi Service (0-5)')

# Online boarding vs satisfaction
sns.countplot(data=train_df, x='Online boarding', hue='satisfaction', ax=axes[1], palette=['#f43f5e', '#10b981'])
axes[1].set_title('Satisfaction by Online Boarding (0-5)')

plt.tight_layout()
plt.show()


### 5. Preprocessing Pipeline Construction
Setting up ColumnTransformer with:
- **Median Imputation + StandardScaler** for numerical variables
- **Most-Frequent Imputation + OneHotEncoder(handle_unknown='ignore')** for nominal variables


In [ ]:
categorical_cols = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
numeric_cols = [
    'Age', 'Flight Distance', 'Inflight wifi service', 'Departure/Arrival time convenient',
    'Ease of Online booking', 'Gate location', 'Food and drink', 'Online boarding',
    'Seat comfort', 'Inflight entertainment', 'On-board service', 'Leg room service',
    'Baggage handling', 'Checkin service', 'Inflight service', 'Cleanliness',
    'Departure Delay in Minutes', 'Arrival Delay in Minutes'
]
target_col = 'satisfaction'

y_train = (train_df[target_col] == 'satisfied').astype(int)
X_train = train_df[numeric_cols + categorical_cols]

y_test = (test_df[target_col] == 'satisfied').astype(int)
X_test = test_df[numeric_cols + categorical_cols]

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, numeric_cols),
    ('cat', cat_pipeline, categorical_cols)
])

print('Preprocessor defined successfully.')


### 6. Logistic Regression Model Training
Fitting complete classification pipeline on training records without data leakage.


In [ ]:
model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs'))
])

print('Training Logistic Regression pipeline...')
model_pipeline.fit(X_train, y_train)
print('Training finished!')


### 7. Evaluation on Held-Out Test Set
Generating evaluation metrics: Accuracy, Precision, Recall, F1-Score, and ROC-AUC.


In [ ]:
y_pred = model_pipeline.predict(X_test)
y_prob = model_pipeline.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f'Test Accuracy:  {acc*100:.2f}%')
print(f'Test Precision: {prec*100:.2f}%')
print(f'Test Recall:    {rec*100:.2f}%')
print(f'Test F1-score:  {f1*100:.2f}%')
print(f'Test ROC-AUC:   {auc:.4f}')
print('\nClassification Report:\n', classification_report(y_test, y_pred, target_names=['Neutral/Dissatisfied', 'Satisfied']))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
            xticklabels=['Neutral/Dissat', 'Satisfied'], yticklabels=['Neutral/Dissat', 'Satisfied'])
axes[0].set_title('Test Confusion Matrix')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='#2563eb', lw=2.5, label=f'Logistic Regression (AUC = {auc:.3f})')
axes[1].plot([0, 1], [0, 1], color='#94a3b8', linestyle='--', label='Random Baseline (AUC = 0.500)')
axes[1].set_title('Receiver Operating Characteristic (ROC) Curve')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()


### 8. Logistic Regression Feature Coefficients
Inspecting model weights across normalized inputs.


In [ ]:
encoder = model_pipeline.named_steps['preprocessor'].named_transformers_['cat'].named_steps['encoder']
cat_feature_names = encoder.get_feature_names_out(categorical_cols).tolist()
all_features = numeric_cols + cat_feature_names
coefs = model_pipeline.named_steps['classifier'].coef_[0]

coef_df = pd.DataFrame({'Feature': all_features, 'Coefficient': coefs})
coef_df['Abs_Impact'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values(by='Abs_Impact', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=coef_df.head(15), x='Coefficient', y='Feature', palette='vlag')
plt.title('Top 15 Feature Coefficients in Logistic Regression')
plt.xlabel('Logistic Regression Coefficient (Impact on Log-Odds)')
plt.show()


### 9. Saving Model Pipeline
Exporting the end-to-end trained model pipeline using Joblib.


In [ ]:
os.makedirs('models', exist_ok=True)
model_path = os.path.join('models', 'airline_satisfaction_model.pkl')
joblib.dump(model_pipeline, model_path)
print(f'Model pipeline saved to: {model_path}')
